In [1]:
from __future__ import division
import pandas as pd
import numpy as np
import time
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator

from matplotlib.ticker import FormatStrFormatter

import scipy.stats as sstat
import scipy.signal as ssig
import h5py
from mpl_toolkits.mplot3d import Axes3D
import os
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA as sklearnPCA
import re

# import ephys_unit_analysis as ena
import mz_ephys_unit_analysis as mz_ena

#import resampy
import fnmatch
import seaborn as sns
%matplotlib inline
%load_ext autoreload
%autoreload 2

---

# NP probe inserted through V1 and hippo 

In [2]:
probe = 'Neuropixels'
channel_groups = mz_ena.get_channel_depth(probe)

In [4]:
root_files = []
matches = [] # list of experiment folders
source_folder = r"G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED"

for root, dirnames, filenames in os.walk(source_folder):
    for filename in fnmatch.filter(filenames, '*rez.mat'):
        for filename in fnmatch.filter(filenames, '*cluster_group.tsv'):#For newer phy2 GUI, .tsv instead of .csv files
            
            # change this before running otherwise there will be none
            if str('sf') in root: 
                if (str('bad') not in root):
                    matches.append(os.path.join(root, filename))
                    root_files.append(root)
                    print (root)

print('\nIMPORTANT: This has "cluster_group.tsv" already appended to the matches list')
print('How many files?', len(matches))
print('How many root files?', len(root_files))

G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED\presf_tuning_cc17995_hp1_2022-07-06_11-36-37
G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED\presf_tuning_cc17995_hp2_2022-07-06_14-44-30
G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED\presf_tuning_cc17988_hp1_2022-07-07_10-30-06
G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED\presf_tuning_cc17988_hp2_2022-07-07_12-41-56
G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED\presf_tuning_cc17952_hp1_2022-07-08_10-28-12
G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED\presf_tuning_cc17952_hp2_2022-07-08_12-38-37
G:\Neuropixels\Pre_tuning_recordings\sf_tuning\SORTED\presf_tuning_cc17952_hp11_2022-07-08_10-52-21

IMPORTANT: This has "cluster_group.tsv" already appended to the matches list
How many files? 7
How many root files? 7


In [5]:



all_units_or_good = 1   # if 0--manually sorted good units, if 1--all units from KS




In [6]:
# run this to check and make sure the splitting in the function below is correct!
f = root_files[0]

f.split('\\')[-1].split('_')

['presf', 'tuning', 'cc17995', 'hp1', '2022-07-06', '11-36-37']

In [7]:
data_df = []
df_rez = []

for path in root_files:
    cluster_path = os.path.join(path, 'cluster_KSLabel.tsv')    
    #------------------------------------------------------------------------------------
    # These probably change depending on the file naming I used during the recording
    rew_type = path.split('\\')[-1].split('_')[0]
    CC_num = path.split('\\')[-1].split('_')[2]   # what cage number is this from?
    HP_num = path.split('\\')[-1].split('_')[3]   # what hp# is the mouse?
    #------------------------------------------------------------------------------------
    cluster_groups = pd.read_csv(cluster_path, sep = '\t')
    #------------------------------------------------------------------------------------
    if all_units_or_good == 0:
        good = cluster_groups[cluster_groups['group'] == 'good'].cluster_id.values
    elif all_units_or_good == 1:
        good = cluster_groups[cluster_groups['KSLabel'] == 'good'].cluster_id.values
    #------------------------------------------------------------------------------------
    spike_clusters = np.load(os.path.join(path, 'spike_clusters.npy'))
    spike_times = np.load(os.path.join(path, 'spike_times.npy'))
    templates = np.load(os.path.join(path, 'templates.npy'))
    spike_templates = np.load(os.path.join(path, 'spike_templates.npy'))
    #------------------------------------------------------------------------------------
    foo = pd.DataFrame({'rew_type':rew_type,
                            'CC':CC_num,
                            'HP':HP_num,
                            'et': CC_num + '_'  + HP_num,
                            'cluster_id':spike_clusters.flatten(),
                            'times':spike_times.flatten()/30000.0, 
                            'templates':spike_templates.flatten(),
                            'path':f})        
    data_df.append(foo)
    #------------------------------------------------------------------------------------
    foo_1 = foo[foo.cluster_id.isin(good)]
    df_rez.append(foo_1)
    #------------------------------------------------------------------------------------
data_df = pd.concat(data_df, axis=0, ignore_index=True)
df_rez = pd.concat(df_rez, axis=0, ignore_index=True)

print('total units df shape:', data_df.shape)
print('"good" units df shape:', df_rez.shape)


total units df shape: (5989704, 8)
"good" units df shape: (3188107, 8)


In [8]:
data_df['cuid'] =  data_df.et.astype(str) + str('_') + data_df.cluster_id.astype(str)
df_rez['cuid'] =  df_rez.et.astype(str) + str('_') + df_rez.cluster_id.astype(str)

print("Total units:", data_df['cuid'].nunique())
print("Good units:", df_rez['cuid'].nunique())

df_rez.head()

Total units: 5233
Good units: 2462


,rew_type,CC,HP,et,cluster_id,times,templates,path,cuid
0,presf,cc17995,hp1,cc17995_hp1,494,0.003467,494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_494
1,presf,cc17995,hp1,cc17995_hp1,171,0.003667,171,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_171
2,presf,cc17995,hp1,cc17995_hp1,579,0.003867,579,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_579
3,presf,cc17995,hp1,cc17995_hp1,3,0.004067,3,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_3
4,presf,cc17995,hp1,cc17995_hp1,117,0.004300,117,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_117


---

# Reward - discrimination training
- rewH20 = rewarded stimulus where water is given
- rewnoH20 = rewarded stimulus where no water is given
- unrew = unrewarded stimulus

In [ ]:
#Psuedo random presentation of the stimuli - 25 per row * 6 rows = 150 trials
# 0 -- drifting grating, rewarded stimulus, 100 trials
# 1 -- pink noise, unrewarded stimulus, 50 trials
stim_order = [0,1,1,0,0,1,0,0,1,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,
               0,1,0,1,0,1,0,0,1,1,0,0,0,1,0,1,0,0,1,1,0,0,0,0,1,
               0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,1,0,0,0,0,1,
               0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,
               1,0,0,0,1,0,1,0,0,0,1,0,1,0,0,1,0,0,0,1,0,1,1,0,0,
               0,0,1,0,0,1,0,0,1,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0]

#Psuedo random distribution of water to the rewarded stimuli
# 0 -- water given -- 80 times
# 1 -- no water given -- 20 times
rew_order = [0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,
               0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,
               0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,
               0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0]

tot_trials = 150

overall_order = []
i=0
for idx,val in enumerate(stim_order):
    if val == 0:
        i = i+1
        if rew_order[i-1] == 0:
            overall_order.append('0') # rewH20
        elif rew_order[i-1] == 1:
            overall_order.append('1') # rewnoH20
    elif val == 1:
        overall_order.append('2')     # unrew
print(len(overall_order))

# Orientation tuning

In [ ]:
angle_iteration = 30 # 30 for 12, 45 for 8 directions

tot_trials = 180

# for 12 orientations
overall_order = [10, 7, 3, 2, 4, 8, 9, 5, 7, 3, 4, 8, 3, 2, 1, 
                 8, 0, 4, 9, 11, 10, 9, 1, 11, 4, 0, 7, 1, 2, 8, 
                 2, 9, 11, 9, 6, 5, 10, 4, 9, 0, 7, 11, 9, 5, 9, 
                 10, 11, 6, 8, 9, 5, 4, 2, 8, 11, 2, 10, 3, 5, 1,
                 7, 0, 4, 9, 1, 5, 11, 3, 5, 10, 1, 2, 9, 6, 2,
                 2, 11, 5, 10, 7, 3, 7, 4, 6, 8, 4, 1, 8, 0, 11,
                 0, 6, 2, 11, 1, 10, 3, 8, 3, 1, 2, 10, 5, 3, 11, 
                 1, 7, 3, 4, 7, 8, 4, 6, 7, 11, 7, 0, 8, 6, 10, 
                 4, 5, 7, 2, 10, 3, 5, 9, 8, 6, 3, 2, 0, 11, 0,
                 6, 10, 0, 7, 4, 5, 0, 10, 6, 8, 10, 3, 11, 9, 0, 
                 5, 1, 3, 7, 0, 6, 9, 1, 6, 10, 5, 6, 11, 7, 0, 
                 5, 1, 4, 1, 6, 8, 2, 9, 2, 8, 3, 0, 4, 6, 1]

# for 8 orientations
# overall_order = [2, 0, 7, 5, 6, 3, 3, 1, 4, 5, 4, 7, 4, 7, 5, 2, 2, 2, 2, 1,
#                  1, 7, 5, 3, 1, 1, 3, 4, 2, 3, 6, 6, 0, 5, 1, 2, 7, 6, 5, 3, 
#                  5, 4, 5, 6, 5, 4, 4, 6, 6, 4, 0, 2, 7, 7, 0, 4, 5, 1, 2, 4,
#                  3, 3, 4, 5, 7, 5, 4, 7, 2, 4, 5, 7, 0, 1, 2, 2, 3, 6, 1, 3,
#                  5, 1, 5, 1, 2, 6, 0, 1, 7, 1, 0, 6, 7, 6, 5, 3, 3, 2, 5, 7, 
#                  0, 4, 3, 0, 1, 4, 1, 3, 5, 2, 0, 1, 6, 0, 5, 7, 1, 7, 3, 0, 
#                  6, 7, 0, 2, 5, 2, 3, 7, 4, 3, 3, 6, 3, 1, 2, 1, 5, 7, 6, 0, 
#                  3, 3, 0, 4, 0, 7, 1, 2, 0, 3, 0, 0, 6, 5, 2, 5, 4, 2, 7, 0,
#                  1, 3, 2, 7, 6, 6, 1, 0, 1, 5, 6, 0, 3, 2, 6, 4, 7, 0, 7, 4, 
#                  6, 5, 4, 1, 6, 2, 4, 4, 2, 4, 7, 6, 0, 4, 3, 1, 6, 6, 0, 7]

# Spatial Frequency tuning

In [9]:
overall_order = [2, 0, 0, 4, 3, 3, 0, 4, 4, 3, 2, 5, 3, 2, 0, 4, 1, 0, 5, 2,
                 0, 1, 0, 1, 3, 5, 2, 5, 1, 2, 0, 5, 2, 3, 5, 1, 0, 4, 3, 2, 
                 5, 5, 3, 5, 2, 0, 3, 3, 0, 3, 4, 5, 4, 1, 4, 0, 1, 5, 4, 1,
                 5, 3, 3, 5, 3, 3, 2, 3, 2, 1, 1, 5, 1, 4, 1, 2, 3, 2, 4, 2,
                 1, 0, 5, 5, 2, 2, 4, 1, 4, 1, 3, 1, 0, 4, 4, 4, 4, 0, 5, 4,
                 4, 0, 3, 5, 5, 2, 1, 3, 4, 1, 5, 0, 2, 2, 0, 0, 1, 0, 2, 1]

tot_trials = 120

sf_list = [0.01, 0.02, 0.04, 0.08, 0.16, 0.32]

---

# Keep going from here

In [10]:

trials_number = tot_trials # ~~~~~~~~~~~~~~~~~~~~~~ IMPORTANT THIS IS CORRECT ~~~~~~~~~~~~~~~~~~~~~~

trial_length = 3.0 #this is the length of the recording in OpenEphys (the yellow highlight)
th_bin = 0.01

ls_rawcount = []
ls_lowspikecount = []
ls_refract_violators = []
ls_lowamp_waveforms = []

### Run these to add a stim column (operant, sf-tuning, ori-tuning)

In [11]:
data_df['trial']=(data_df.times//trial_length).astype(int)
data_df['stim']=data_df.trial.map(dict(zip(np.arange(trials_number),([int(i) for i in overall_order]))))

df_rez['trial']=(df_rez.times//trial_length).astype(int)
df_rez['stim']=df_rez.trial.map(dict(zip(np.arange(trials_number),([int(i) for i in overall_order]))))
df_rez.head()

## For recordings where only 1 stimulus is shown (aka novel recordings, or pre, or post training)
# adding the "stim" column keeps the rest of the code able to run
# data_df['trial']=(data_df.times//trial_length).astype(int)
# data_df['stim']=0
# df_rez['trial']=(df_rez.times//trial_length).astype(int)
# df_rez['stim']=0
# df_rez.head()

,rew_type,CC,HP,et,cluster_id,times,templates,path,cuid,trial,stim
0,presf,cc17995,hp1,cc17995_hp1,494,0.003467,494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_494,0,2
1,presf,cc17995,hp1,cc17995_hp1,171,0.003667,171,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_171,0,2
2,presf,cc17995,hp1,cc17995_hp1,579,0.003867,579,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_579,0,2
3,presf,cc17995,hp1,cc17995_hp1,3,0.004067,3,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_3,0,2
4,presf,cc17995,hp1,cc17995_hp1,117,0.004300,117,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,cc17995_hp1_117,0,2


---

# Creating the spikes and tmt dataFrames

In [12]:
ls_spikes = []
ls_tmt = []

i=0

num_units = df_rez['cuid'].nunique()

for iii, unit in enumerate(df_rez['cuid'].unique()): ##### I changed this from df_rez to data_df to check the units
    cuid = str(unit)
    tmp2 = df_rez[(df_rez.cuid == unit)]             ##### I changed this from df_rez to data_df to check the units
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    cluster_id = tmp2.cluster_id.values[0]
    et = tmp2.et.values[0]
    cc = tmp2.CC.values[0]
    path = tmp2.path.values[0]
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    try:
        for stim_id in tmp2.stim.unique():
            tmp3=tmp2[tmp2.stim==stim_id]
            tmt, depth, ch_idx = mz_ena.ksort_get_tmt(tmp3, cluster_id, templates, channel_groups)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
            df = mz_ena.getRaster_kilosort(tmp3, unit, trial_length) 
            trials_number_not_empty = len(df.trial.unique())    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
            df_spikes_tmp = pd.DataFrame({'cluster_id': cluster_id, 
                                          'spikes': tmp3.times.values,
                                          'trial':df.trial,
                                          'stim_id':stim_id,
                                          'trial_spikes':df.times,
                                          'depth':depth,
                                          'et':et,
                                          'cc': cc,
                                          'cuid': cuid,
                                          'path':path})
            ls_spikes.append(df_spikes_tmp)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
            df_tmt_tmp = pd.DataFrame({'tmt': tmt,
                                       'stim_id':stim_id,
                                       'depth':depth,
                                       'et':et,
                                       'cc': cc,
                                       'cuid': cuid,
                                       'path':path})
            ls_tmt.append(df_tmt_tmp)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    except:
        i+=1
        continue
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    if iii%200 == 0:
        print('done with {0} out of {1}'.format(iii, num_units))
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_spikes = pd.concat(ls_spikes)
df_tmt = pd.concat(ls_tmt)
print("Total errors: {0} out of {1} units".format(i,num_units))
print("ALL DONE!")

done with 0 out of 2462
done with 200 out of 2462
done with 400 out of 2462
done with 600 out of 2462
done with 1000 out of 2462
done with 1200 out of 2462
done with 1400 out of 2462
done with 1600 out of 2462
done with 1800 out of 2462
done with 2000 out of 2462
done with 2200 out of 2462
done with 2400 out of 2462
Total errors: 152 out of 2462 units
ALL DONE!


In [13]:
df_spikes.head(2)

,cluster_id,spikes,trial,stim_id,trial_spikes,depth,et,cc,cuid,path
0.0,494,0.003467,0.0,2,0.003467,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
0.0,494,0.148833,0.0,2,0.148833,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...


In [14]:
df_tmt.head(2)

,tmt,stim_id,depth,et,cc,cuid,path
0,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
1,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...


# Function to add metadata

In [15]:
def label_group(row):
    if (row['cc'] == "CC082263") | (row['cc'] == "CC067489") | (row['cc'] == "CC082260") | (row['cc'] == "CC084621"):
        return "A"
    elif (row['cc'] == "CC082257") | (row['cc'] == "CC067431") | (row['cc'] == "CC067432") | (row['cc'] == "CC082255"):
        return "B"

def label_group2(row):
    if (row['cc'] == "cc17995"):
        return "A"
    elif (row['cc'] == "cc17988") | (row['cc'] == "cc17952"):
        return "B"

def label_set(row):
    if (row['cc'] == "CC082263") | (row['cc'] == "CC067489") | (row['cc'] == "CC082257") | (row['cc'] == "CC067431"):
        return "1"
    elif (row['cc'] == "CC082260") | (row['cc'] == "CC084621") | (row['cc'] == "CC067432") | (row['cc'] == "CC082255"):
        return "2"

def label_region(row):
    if (row['depth'] <= 3050) & (row['depth'] >= 1900):
        return 'v1'
    elif (row['depth'] < 1800) & (row['depth'] >= 600):
        return 'hippo'
    elif (row['depth'] < 600):
        return 'thal'
    else:
        return 'none'

In [16]:
# df_spikes['group'] = df_spikes.apply(lambda row: label_group(row), axis=1)
df_spikes['group'] = df_spikes.apply(lambda row: label_group2(row), axis=1)
df_spikes['set'] = df_spikes.apply(lambda row: label_set(row), axis=1)
df_spikes['region'] = df_spikes.apply(lambda row: label_region(row), axis=1)

df_spikes.head(2)

,cluster_id,spikes,trial,stim_id,trial_spikes,depth,et,cc,cuid,path,group,set,region
0.0,494,0.003467,0.0,2,0.003467,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1
0.0,494,0.148833,0.0,2,0.148833,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1


In [17]:
# df_tmt['group'] = df_tmt.apply(lambda row: label_group(row), axis=1)
df_tmt['group'] = df_tmt.apply(lambda row: label_group2(row), axis=1)
df_tmt['set'] = df_tmt.apply(lambda row: label_set(row), axis=1)
df_tmt['region'] = df_tmt.apply(lambda row: label_region(row), axis=1)

df_tmt.head(2)

,tmt,stim_id,depth,et,cc,cuid,path,group,set,region
0,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1
1,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1


---

# Adding to the waveforms (tmt) dataFrame
Based on Yu's code and outputs the spike width, peak-to-trough amplitude, and the overall rs/fs classification

In [18]:
import resampy
import scipy
import scipy.signal as ssig

def gaus(x,a,x0,sigma):
    return a*np.exp(-(x-x0)**2/(2*sigma**2))

result = df_tmt

In [19]:
spk_width = {}
tr2peak = {}
neuron_type = {}
tp_dic = {}
w_dic = {}
ls = []
ls2 = []
ls3 = []

num_units = result['cuid'].nunique()

for ii,cuid in enumerate(result.cuid.unique()[:]):    
    if ii%200 == 0:
        print('done with {0} out of {1}'.format(ii, num_units))
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    tmt_data = np.array(result[(result['cuid'] == cuid)].tmt)

    y = resampy.resample( tmt_data[::-1] , 1 ,10,  filter='sinc_window',
                                    num_zeros=10, precision=5, window=ssig.hann)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #trough-to-peak
    trough_idx = y.argmin()
    peak_idx = y[:y.argmin()].argmax()
    tp = abs((trough_idx - peak_idx)/300.0)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    x = np.arange(y.size)
    y_gaus = y*(-1)
    popt,pcov = scipy.optimize.curve_fit(gaus,x,y_gaus,p0=[0.2, y.argmin(), 10])
    fwhm = popt[-1]/300*2.355
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #width of spike    
    f,pxx = ssig.welch(tmt_data, fs=30000,  nfft=5096,  nperseg=48,
                          return_onesided=True, scaling='spectrum')

    df = np.vstack((f, pxx))
    df = pd.DataFrame(df)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    idx = df.T[1].idxmax()
    if not np.isnan(idx):
        w = df.T[0][idx]
        w = 1/w*1000.0
        ls.append(tp)
        ls2.append(w)
        spk_width[cuid] = w
        tr2peak[cuid] = tp
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        if tp<=0.45: #can add something for width here as well, but then you'll have 'fs', 'rs', and 'un' units
            neuron_type[cuid] = 'fs'
        elif tp>0.45:
            neuron_type[cuid] = 'rs'
        else:
            neuron_type[cuid] = 'un'
        #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        # p/t ratio
        ptr = result[(result['cuid'] == cuid)]['tmt'].max()/result[(result['cuid'] == cuid)]['tmt'].min()
        ls3.append(abs(ptr))
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~        
df_tmt['n_type'] = df_tmt.cuid.map(neuron_type)
df_tmt['trough2peak'] = df_tmt.cuid.map(tr2peak)
df_tmt['spk_width'] = df_tmt.cuid.map(spk_width)

df_tmt.head()

done with 0 out of 2310
done with 200 out of 2310
done with 400 out of 2310
done with 600 out of 2310
done with 800 out of 2310
done with 1000 out of 2310
done with 1200 out of 2310
done with 1400 out of 2310
done with 1600 out of 2310
done with 1800 out of 2310
done with 2000 out of 2310
done with 2200 out of 2310


,tmt,stim_id,depth,et,cc,cuid,path,group,set,region,n_type,trough2peak,spk_width
0,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1,rs,0.54,1.316796
1,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1,rs,0.54,1.316796
2,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1,rs,0.54,1.316796
3,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1,rs,0.54,1.316796
4,0.0,2,2140,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1,rs,0.54,1.316796


In [20]:
print('Number of rs units: {0}'.format(df_tmt[df_tmt['n_type'] == 'rs'].cuid.nunique()))
print('Number of fs units: {0}'.format(df_tmt[df_tmt['n_type'] == 'fs'].cuid.nunique()))
print('Number of un units: {0}'.format(df_tmt[df_tmt['n_type'] == 'un'].cuid.nunique()))

Number of rs units: 1503
Number of fs units: 807
Number of un units: 0


# Reorder the columns

In [21]:
# just a last reordering of the columns for easy viewing -- spikes df
cols = ['cluster_id', 'spikes', 'trial', 'stim_id', 'trial_spikes', 
        'depth', 'cuid', 'group', 'set', 'region', 'et', 'cc','path']

df_spikes = df_spikes[cols]

df_spikes.head()

,cluster_id,spikes,trial,stim_id,trial_spikes,depth,cuid,group,set,region,et,cc,path
0.0,494,0.003467,0.0,2,0.003467,2140,cc17995_hp1_494,A,None,v1,cc17995_hp1,cc17995,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
0.0,494,0.148833,0.0,2,0.148833,2140,cc17995_hp1_494,A,None,v1,cc17995_hp1,cc17995,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
0.0,494,0.361267,0.0,2,0.361267,2140,cc17995_hp1_494,A,None,v1,cc17995_hp1,cc17995,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
0.0,494,0.565667,0.0,2,0.565667,2140,cc17995_hp1_494,A,None,v1,cc17995_hp1,cc17995,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
0.0,494,0.603000,0.0,2,0.603000,2140,cc17995_hp1_494,A,None,v1,cc17995_hp1,cc17995,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...


In [22]:
# just a last reordering of the columns for easy viewing -- waveforms df
cols = ['tmt', 'n_type', 'trough2peak', 'spk_width', 'stim_id', 'cuid', 
        'depth', 'region', 'group', 'set', 'cc', 'et', 'path']

df_tmt = df_tmt[cols]

df_tmt.head()

,tmt,n_type,trough2peak,spk_width,stim_id,cuid,depth,region,group,set,cc,et,path
0,0.0,rs,0.54,1.316796,2,cc17995_hp1_494,2140,v1,A,None,cc17995,cc17995_hp1,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
1,0.0,rs,0.54,1.316796,2,cc17995_hp1_494,2140,v1,A,None,cc17995,cc17995_hp1,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
2,0.0,rs,0.54,1.316796,2,cc17995_hp1_494,2140,v1,A,None,cc17995,cc17995_hp1,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
3,0.0,rs,0.54,1.316796,2,cc17995_hp1_494,2140,v1,A,None,cc17995,cc17995_hp1,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
4,0.0,rs,0.54,1.316796,2,cc17995_hp1_494,2140,v1,A,None,cc17995,cc17995_hp1,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...


# Save the Spikes and Waveform dataFrames

In [23]:

df_spikes.to_pickle(r"D:\mz_Data\saved_dfs\Operant_reward\sf_tuning\pre_sf_spikes_df.pkl")

df_tmt.to_pickle(r"D:\mz_Data\saved_dfs\Operant_reward\sf_tuning\pre_sf_waveforms_df.pkl")


---

# Making the PSTH dataFrame

In [24]:
ls_psth = []
i=0

num_units = df_rez['cuid'].nunique()

for iii, unit in enumerate(df_rez['cuid'].unique()): ##### I changed this from df_rez to data_df to check the units
    cuid = str(unit)
    tmp2 = df_rez[(df_rez.cuid == unit)]             ##### I changed this from df_rez to data_df to check the units
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    cluster_id = tmp2.cluster_id.values[0]
    et = tmp2.et.values[0]
    cc = tmp2.CC.values[0]
    path = tmp2.path.values[0]
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    try:
        tmt, depth, ch_idx = mz_ena.ksort_get_tmt(tmp2, cluster_id, templates, channel_groups)
    except:
        i = i+1        
        continue    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    if iii%200 == 0: #think of this as a loading bar to know for far it has run
        print('done with {0} out of {1}'.format(iii, num_units))
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    for stim_id in tmp2.stim.unique():
        tmp3=tmp2[tmp2.stim==stim_id]
        df = mz_ena.getRaster_kilosort(tmp3, unit, trial_length) 
        trials_number_not_empty = len(df.trial.unique())    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        h, ttr = mz_ena.PSTH(df.times, th_bin, trial_length, trials_number_not_empty)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        zscore = sstat.mstats.zscore(h)
        mean = np.mean(h[0:50])#The Baseline period. Be sure it matches time course of experiments##
        if mean<=0:
            std=1
        else:
            std = np.std(h[0:50])#The Baseline period. Be sure it matches time course of experiments##
        ztc = (h - mean)/std
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        df_psth_tmp = pd.DataFrame({'times':ttr,
                                    'stim_id': stim_id,
                                    'Hz':h,
                                    'cluster_id': cluster_id,
                                    'depth': depth,
                                    'zscore':zscore, 
                                    'ztc':ztc,
                                    'et':et,
                                    'cc': cc,
                                    'cuid':cuid,
                                    'path':path})
        ls_psth.append(df_psth_tmp)
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_psth = pd.concat(ls_psth)
print("Total errors: {0} out of {1} units".format(i,num_units))
print("ALL DONE!")


done with 0 out of 2462
done with 200 out of 2462
done with 400 out of 2462
done with 600 out of 2462
done with 1000 out of 2462
done with 1200 out of 2462
done with 1400 out of 2462
done with 1600 out of 2462
done with 1800 out of 2462
done with 2000 out of 2462
done with 2200 out of 2462
done with 2400 out of 2462
Total errors: 152 out of 2462 units
ALL DONE!


In [25]:
print('Min unit depth on probe:', df_psth['depth'].min())
print('Max unit depth on probe:',df_psth['depth'].max())

# print('\n',np.sort(df_psth['depth'].unique()))

Min unit depth on probe: 20
Max unit depth on probe: 3820


In [26]:
null_df = df_psth[df_psth['zscore'] < 20]

print('# good units: %d' % df_psth['cuid'].nunique())
print('# good units w/ z-score < 20: %d' % null_df['cuid'].nunique())

# good units: 2310
# good units w/ z-score < 20: 2310


---

# Updating the PSTH dataFrame with metadata
- group a: 082263, 067489, 082260, 084621
- group b: 082257, 067431, 067432, 082255

In [27]:
# null_df['group'] = null_df.apply(lambda row: label_group(row), axis=1)
null_df['group'] = null_df.apply(lambda row: label_group2(row), axis=1)
null_df['set'] = null_df.apply(lambda row: label_set(row), axis=1)
null_df['region'] = null_df.apply(lambda row: label_region(row), axis=1)

null_df.head()

,times,stim_id,Hz,cluster_id,depth,zscore,ztc,et,cc,cuid,path,group,set,region
0,0.00,2,2.138713,494,2140,-1.900212,-2.124411,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1
1,0.01,2,2.500587,494,2140,-1.399673,-1.385966,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1
2,0.02,2,2.915890,494,2140,-0.825232,-0.538493,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1
3,0.03,2,3.244291,494,2140,-0.370993,0.131647,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1
4,0.04,2,3.583491,494,2140,0.098183,0.823823,cc17995_hp1,cc17995,cc17995_hp1_494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...,A,None,v1


---

# Reordering the columns

In [28]:
# just a last reordering of the columns for easy viewing -- Reward trials
cols = ['stim_id', 'times', 'cuid', 'depth', 'Hz', 'zscore', 'ztc', 
        'region', 'group', 'set', 'cc', 'et', 'cluster_id', 'path']
null_df = null_df[cols]
null_df.head()

,stim_id,times,cuid,depth,Hz,zscore,ztc,region,group,set,cc,et,cluster_id,path
0,2,0.00,cc17995_hp1_494,2140,2.138713,-1.900212,-2.124411,v1,A,None,cc17995,cc17995_hp1,494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
1,2,0.01,cc17995_hp1_494,2140,2.500587,-1.399673,-1.385966,v1,A,None,cc17995,cc17995_hp1,494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
2,2,0.02,cc17995_hp1_494,2140,2.915890,-0.825232,-0.538493,v1,A,None,cc17995,cc17995_hp1,494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
3,2,0.03,cc17995_hp1_494,2140,3.244291,-0.370993,0.131647,v1,A,None,cc17995,cc17995_hp1,494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...
4,2,0.04,cc17995_hp1_494,2140,3.583491,0.098183,0.823823,v1,A,None,cc17995,cc17995_hp1,494,G:\Neuropixels\Pre_tuning_recordings\sf_tuning...


---

# Save the dataframe and plot elsewhere

In [29]:


null_df.to_pickle(r"D:\mz_Data\saved_dfs\Operant_reward\sf_tuning\pre_sf_psth_df.pkl")



---